# Pandas

* Jake VanderPlas. 2016. *Python Data Science Handbook: Essential Tools for Working with Data*. O'Reilly Media, Inc.
* Chapter 3 - Data Manipulation with Pandas
* https://github.com/jakevdp/PythonDataScienceHandbook

Pandas provides:

* Rich I/O Capabilities (read/write data from/to CSV, Excel, SQL, JSON, etc.)
* 1-dimensional (**Series**) and 2-dimensional tabular (**DataFrame**) data structures.
* Data flexibility (handles missing data, time series, and heterogeneous data types).
* Labeled Rows and columns for data alignment
* Flexible indexing, slicing, fancy indexing, and subsetting of large datasets.

In [ ]:
import numpy as np
import pandas as pd

pd.__version__

In [ ]:
# type TAB to get the numpy namespace
#pd.

In [ ]:
def pprint(*args, sep='\n', end='\n', userepr=True, align=False, breakline=False, indent='   '):
    "Evaluate and pretty print"
    if align :
        max_arg_len = max(map(len,args))
        txt2txt = lambda txt : ' '*(max_arg_len-len(txt)) + txt
        args = list(map(txt2txt, args))
    geval = lambda txt : eval(txt, globals())
    txt2txt = lambda txt : f'{repr(geval(txt))}' if userepr else f'{geval(txt)}'    
    output = map(txt2txt, args)
    txt2txt = lambda txt : f'{txt} = '    
    prefix = map(txt2txt, args)
    if breakline :
        po2txt = lambda p,o : (p+'\n'+o).replace('\n','\n'+indent)
    else:
        po2txt = lambda p,o : (p+o).replace('\n','\n'+' '*len(p))
    print(*map(po2txt, prefix, output), sep=sep, end=end)

class display(object):
    """Display HTML representation of multiple objects"""
    template = """<div style="float: left; padding: 10px;">
    <p style='font-family:"Courier New", Courier, monospace'>{0}</p>{1}
    </div>"""
    def __init__(self, *args):
        self.args = args
        
    def _repr_html_(self):
        return '\n'.join(self.template.format(a, eval(a)._repr_html_())
                         for a in self.args)
    
    def __repr__(self):
        return '\n\n'.join(a + '\n' + repr(eval(a))
                           for a in self.args)

from IPython.display import HTML
HTML("""
<style>
.dataframe {
    font-size: 80% !important; /* Adjust the percentage as needed */
}
</style>
""")

## Pandas Series

* One-dimensional array of indexed data
* Two key attributes:
   * `values` : NumPy array
   * `index` : an array-like object of type `pd.Index`

In [ ]:
d = pd.Series([0.25, 0.5, 0.75, 1.0])
#print(f'{d=}\n{d.values=}\n{d.index}')
pprint('d', 'd.values', 'd.index', align=True)

Series can be created from NumPy arrays:

In [ ]:
d = pd.Series(np.linspace(0,4,6))
d

A series can be indexed just like a NumPy array:

In [ ]:
d = pd.Series(np.arange(10,15))
pprint('d[3]')  # simple index --> scalar
pprint('d[3:]', 'd[3:4]')  # slice --> series
pprint('d[[1,3,0]]', 'd[[1]]')  # fancy index --> series

But... Pandas Series can have an **explicit index**
   * If not provided, an **implicit int index** is used `[0,1,...,n-1]`

In [ ]:
d = pd.Series(np.arange(10,15), index=["a","b","c","d","e"])
pprint('d')
# WARNING "treating keys as positions is deprecated"  --> use Series.iloc[pos]
#pprint('d[1]')
pprint('d["b"]')

### Slicing - Case 1 - non-integer explicit index
   * Both non-integer (explicit) and integer (implicit) slices can be used.
   * non-integer slice &rarr; **label-based indexing** &rarr; $[from,to]$
   * integer slice &rarr; **position-based indexing** &rarr; $[from,to)$

<br>

In [ ]:
d = pd.Series([100,101,102,103,104], index=["a","b","c","d","e"])
pprint('d')
pprint('d["a"]', 'd["c"]', 'd["a":"c"].values')   # explicit index
pprint('d[0:2].values')                           # implicit index

### Slicing - Case 2 - integer explicit index
   * Slicing is <u>always referred to implicit indices</u>.
   * integer slice &rarr; **position-based indexing** &rarr; $[from,to]$

<br>

In [ ]:
d = pd.Series([100,101,102,103,104], index=[1,5,4,2,3])
pprint('d')
pprint('d[1]', 'd[3]')  # explicit index
pprint('d[1:3].values') # implicit index

### Fancy Indexing
   * Label-based indexing
   * Using <u>position-based indexing</u> is **deprecated** 

<br>

In [ ]:
d = pd.Series([100,101,102,103,104], index=["a","b","c","d","e"])
pprint('d[["a","b","c"]].values')   # explicit index
# WARNING "treating keys as positions is deprecated"  --> use Series.iloc[pos]
#pprint('d[[1,2,3]].values')   # implicit index

d = pd.Series([100,101,102,103,104], index=[1,5,4,2,3])
pprint('d[[1,2,3]].values')   # explicit index

### Series and dictionaries
* Series are kind of specialized Python dictionaries $\{index_{typed \,\&\, ordered} \to value_{typed}\}$ 
* `pd.Series(dict)` &rarr; create a Series from a dictionary
* `pd.Series.to_dict()` &rarr; create a dictionary from a Series<br><br>

In [ ]:
d1 = pd.Series([100,101,102], index=["a","b","c"])
x = d1.to_dict()
d2 = pd.Series(x)
pprint('d1','x','d2')

* `.keys()` &rarr; `.index` <br><br>

In [ ]:
d = pd.Series([100,101,102,103,104], index=["a","b","c","d","e"])
pprint('type(d.index) == type(d.keys())')
pprint('d.index == d.keys()')
pprint('d.index is d.keys()')

 <br>
 
 * $\nexists\;\;$ `.values()`

In [ ]:
d.values
# TypeError: 'numpy.ndarray' object is not callable
#d.values()

* `.items()` &rarr; `zip` object of `(index,value)` <br><br>

In [ ]:
d = pd.Series([100,101,102,103,104], index=["a","b","c","d","e"])
pprint('d.items()')
print(type(d.items()))
print(*d.items())


<br>

* `x in series` &rarr; checks if Series index contains `x`

In [ ]:
d = pd.Series([100,101,102,103,104], index=["a","b","c","d","e"])
pprint('102 in d', '"a" in d')

## Pandas DataFrame

* DataFrames are kind of specialized Python dictionaries $\{index_{typed \,\&\, ordered} \to series\}$ with a common row index
* Kind of *Series of Series* with a common row index
* * Three key attributes:
   * `values` : NumPy array
   * `index` and `columns`: array-like objects of type `pd.Index`

<br>

In [ ]:
cities = ['California','Texas','Florida','New York']
population = pd.Series([39538223,29145505,21538187,20201249], index=cities)
area = pd.Series([423967, 695662, 170312, 141297], index=cities)
#pprint('population', 'area')
states = pd.DataFrame({'population':population, 'area':area})
states

* The `index` attribute (*row index*) and `columns` attribute (*column index*) are of type `pd.Index`
* The `values` attribute (of type `np.ndarray`) contains the data

<br>

In [ ]:
states.index

<br>

In [ ]:
states.columns

<br>

In [ ]:
states.values

A DataFrame can be constructed from a single series:<br><br>

In [ ]:
cities = ['California','Texas','Florida','New York']
population = pd.Series([39538223,29145505,21538187,20201249], index=cities)
states = pd.DataFrame(population, columns=['population'])
states

A DataFrame can be constructed from a dictionary of Series:<br><br>

In [ ]:
cities = ['California','Texas','Florida','New York']
population = pd.Series([39538223,29145505,21538187,20201249], index=cities)
area = pd.Series([423967, 695662, 170312, 141297], index=cities)
states = pd.DataFrame({'population':population, 'area':area})
states

A DataFrame can be constructed from list of dictionaries:<br><br>

In [ ]:
data = [
    {'population':39538223, 'area':423967},
    {'population':29145505, 'area':695662},
    {'population':21538187, 'area':170312},
    {'population':20201249, 'area':141297}
]
states = pd.DataFrame(data, index=["California","Texas","Florida","New York"])
states

A DataFrame can be constructed from a two-dimensional NumPy array:<br><br>

In [ ]:
data = np.array([[39538223,   423967],
          [29145505,   695662],
          [21538187,   170312],
          [20201249,   141297]])
states = pd.DataFrame(data,
                      columns=["population","area"],
                      index=["California","Texas","Florida","New York"])
states

A DataFrame can integrate indices in different orders<br><br>

In [ ]:
cities1 = ['California','Texas','Florida','New York']
population = pd.Series([39538223,29145505,21538187,20201249], index=cities1)
cities2 = ['Texas','Florida','New York','California']
area = pd.Series([695662, 170312, 141297, 423967], index=cities2)
#pprint('population','area', align=True)
states = pd.DataFrame({'population':population, 'area':area})
states

A DataFrame can integrate indices with different values
* *Missing* values are filled with `NaN`s <br><br>

In [ ]:
cities1 = ['California','Texas','Florida']
population = pd.Series([39538223,29145505,21538187], index=cities1)
cities2 = ['Texas','Florida','New York']
area = pd.Series([695662, 170312, 141297], index=cities2)
#pprint('population','area', align=True)
states = pd.DataFrame({'population':population, 'area':area})
states

## Pandas Index
* Kind of Immutable NumPy array
* Many NumPy attributes, indexable, sliceable, etc. <br><br>

In [ ]:
ind = pd.Index(['California', 'Texas', 'Florida'])
pprint('ind', 'ind[1]', 'ind[:2]', 'ind[[2,0,1]]', align=True, end='\n\n')
pprint('ind.size', 'ind.shape', 'ind.ndim', 'ind.dtype', align=True)

In [ ]:
ind = pd.Index([200, 1000, 300])
pprint('ind', 'ind[1]', 'ind[:2]', 'ind[[2,0,1]]', align=True, end='\n\n')
pprint('ind.size', 'ind.shape', 'ind.ndim', 'ind.dtype', align=True)

`pd.Index` follows many of the conventions of Python's `set`

* `idx1.union(idx2)` &rarr; combines elements from both Index
* `idx1.intersection(idx2)` &rarr; elements that are common to both Index
* `idx1.difference(idx2)` &rarr; elements in `idx1` that are not in `idx2`
* `idx1.symmetric_difference(idx2)` &rarr; elements in either, but not in both
* `idx1.equals(idx2)` &rarr; checks if they contain the same elements in the same order

<br>

In [ ]:
idx1 = pd.Index([1,2,3,4,5])
idx2 = pd.Index([4,5,6,7])
pprint('idx1.union(idx2)', 'idx1.intersection(idx2)',
       'idx1.difference(idx2)', 'idx1.symmetric_difference(idx2)',
       'idx1.equals(idx2)', align=True)

## Pandas I/O
* Powerful and flexible IO collection of functions
   * Text-based files: CSV, TSV, plain text
   * Spreadsheet files: Excel, OpenOffice/LibreOffice
   * Web-based data: HTML
   * Structured data formats: JSON, XML
   * Relational Databases: SQL
   * Other formats: Pickle, HDF5, Parquet, ...
* In (`pd.read_...`): `pd.read_csv`, `pd.read_excel`, `pd.read_json`,... `
* Out (`df.to_...`): `df.to_csv`, `df.to_excel`, `df.to_json`,... `


In [ ]:
#pd.read_
#states.to_

In [ ]:
url = "https://raw.githubusercontent.com/pandas-dev/pandas/refs/heads/main/pandas/tests/io/data/csv/iris.csv"
pd.read_csv(url)

In [ ]:
#!pip install openpyxl
url = "https://github.com/pandas-dev/pandas/raw/refs/heads/main/pandas/tests/io/data/excel/test1.xlsx"
pd.read_excel(url, index_col=0) # use first column as index

## Indexing and Selection

In [ ]:
cities = ['California', 'Texas', 'Florida', 'New York', 'Pennsylvania']
area = pd.Series([423967, 695662, 170312, 141297, 119280], index=cities)
population = pd.Series([39538223, 29145505, 21538187, 20201249, 13002700], index=cities)
gdp = pd.Series([3.9, 2.7, 1.3, 1.8, 1.0], index=cities) # Gross Domestic Product 
df = pd.DataFrame({'area':area, 'population':population, 'GDP':gdp})
df

* Simple Indexing &rarr; column selection (**Series**)

In [ ]:
df['area']
# KeyError: 0
#df[0]

* Fancy Indexing &rarr; columns selection (**DataFrame**)

In [ ]:
display( "df[['GDP','area']]", "df[['area']]")

In [ ]:
pprint("df['GDP']")

* Slicing &rarr; rows selection (**DataFrame**)

In [ ]:
display('df[:2]', 'df[1:2]', 'df[2:2]')

### Combining columns & rows selection
* columns are selected with fancy indexing
* rows are selected with (position) slicing

In [ ]:
display("df[['area','population']][:3]" , "df[:3][['area','population']]")

### Accessing the data through `df.values`
   * `df.values` &rarr; homogeneous NumPy array
   * Series values dtypes can be promoted

In [ ]:
pprint('df.values')
pprint('df.values.dtype', 'df["area"].dtype',
       'df["population"].dtype', 'df["GDP"].dtype', align=True)

### Masking with comparison operator
* Let `op` be an operator
* `Series op value` &rarr; broadcasted operation &rarr; Series
* `DataFrame op value` &rarr; broadcasted operation &rarr; DataFrame<br><br>

In [ ]:
pprint("df['GDP'] > 2")

In [ ]:
display("df * 10" , "df > 2")

Boolean Series/DataFrame can be used as index
* Boolean Series &rarr; select rows
* Boolean DataFrame &rarr; select elements (and fill with `NaN`)

In [ ]:
display("df[df['GDP'] > 2]" , "df[df > 2]")

Find the area for cities with `GDP>1` and `population<30000000`:

In [ ]:
df[df['GDP'] > 1]

In [ ]:
df[(df['GDP']>1) & (df['population']<30_000_000)]

In [ ]:
df[(df['GDP']>1) & (df['population']<30_000_000)]['area']

### Indexers: `.loc` and `.iloc`
* `.loc` &rarr; explicit indexing (and slicing) 
* `.iloc` &rarr; implicit indexing (and slicing)
* NumPy style indexing
   * `.loc[i]` &rarr; row `i` 
   * `.loc[i,j]` &rarr; row `i` , column `j` 
   * Simple Indexing, Fancy Indexing and Slicing

In [ ]:
df

In [ ]:
df.loc['Florida']

In [ ]:
df.loc[['Florida','Texas']]

In [ ]:
df.loc[:'Florida']

In [ ]:
df.iloc[2]

In [ ]:
df.iloc[[2,1]]

In [ ]:
df.iloc[:3]

In [ ]:
df.loc['Florida','population']

In [ ]:
df.loc[['California','Florida'],'population']

In [ ]:
df.loc[['California','Florida'],:]

Using `.iloc` is similar to using `df.values`, but maintaining the DataFrame structure (vs. homogeneous NumPy array)

In [ ]:
df.iloc[:2,[0,2]]

In [ ]:
pprint("df.values[:2,[0,2]]" , "df.values[:2,[0,2]].dtype", align=True)

## Modifying DataFrames Through Indexing
* The indexing can be used to modify a DataFrame
   * Modify values
   * Add/remove columns
   * Add/remove rows

### Modifying DataFrame values

* Simple indexing &rarr; column assignment

In [ ]:
df2 = df.copy()
df2['area'] = 0
df2

* Fancy Indexing &rarr; columns assignment

In [ ]:
df2 = df.copy()
df2[['area','GDP']] = 0
df2

* Slicing &rarr; rows assignment

In [ ]:
df2 = df.copy()
df2[:2] = 0
df2

* Assigning to `df.values[i,j]` does not touch the DataFrame

In [ ]:
df2 = df.copy()
df2.values[:2,:2] = 0
df2

* Series masking (index is a boolean Series) &rarr; rows assignment

In [ ]:
df2 = df.copy()
pprint('df2["GDP"]>2')
df2[df2['GDP']>2] = 0
df2

* DataFrame masking (index is a boolean DataFrame) &rarr; *elements* assignment

In [ ]:
df2 = df.copy()
pprint('df2>2')
df2[df2>2] = 0
df2

* `.loc` and `.iloc` &rarr; *subset* assignment

In [ ]:
df2 = df.copy()
df2.loc[['California','New York'],'population':] = 0
df2

In [ ]:
df2 = df.copy()
df2.iloc[[0,3],1:] = 0
df2

### Adding/Removing columns
   * **Add**: `df[new_index] = value`
   * **Remove**: `del df[index]` or `df.drop(index, axis=1, inplace=True)`

In [ ]:
# add column 'density'
df2 = df.copy()
df2['density'] = df2['population'] / df2['area']
df2

In [ ]:
# remove column 'population'
df2 = df.copy()
del df2['population']
df2

In [ ]:
# remove column 'population'
df2 = df.copy()
df2.drop('population', axis=1, inplace=True)
df2

In [ ]:
# remove columns ['area','population']
df2 = df.copy()
df2.drop(['area','population'], axis=1, inplace=True)
df2

### Adding/Removing rows
   * **Add**: `df.loc[new_index] = value` (WARNING &rarr; homogeneous DataFrame)
   * **Remove**: `df.drop(index, axis=0, inplace=True)`

In [ ]:
# add row 'chicago'
df2 = df.copy()
df2.loc['Chicago'] = [111111,22222222,2.0]
df2

In [ ]:
# remove row 'New York'
df2 = df.copy()
df2.drop('New York', axis=0, inplace=True)
df2

In [ ]:
# remove rows ['California','New York']
df2 = df.copy()
df2.drop(['California','New York'], axis=0, inplace=True)
df2

## Operating on Data
   * Arithmetic operators and NumPy functions can be applied to Series/DataFrames
   * Indices are preserved
   * Binary operations/functions align the indices
      * Can manage incomplete data inserting `NaN`s
   * DataFrame and Series operation have broadcasting property

### Index Preservation

In [ ]:
df

In [ ]:
df * 10

In [ ]:
np.log(df[['area','population']])

Python operators have their equivalent Pandas object methods:

| Python operator | Pandas method(s)                |
|-----------------|---------------------------------|
| `+`             | `add`                           |
| `-`             | `sub`, `subtract`               |
| `*`             | `mul`, `multiply`               |
| `/`             | `truediv`, `div`, `divide`      |
| `//`            | `floordiv`                      |
| `%`             | `mod`                           |
| `**`            | `pow`                           |

### Index Alignment

In [ ]:
x = pd.Series({'A': 10,'B': 20})
y = pd.Series({'B': 4, 'A': 5})
x / y 

In [ ]:
x = pd.Series({'A': 10,'B': 20, 'C':30})
y = pd.Series({'B': 4, 'C': 5, 'D':3})
x / y 

In [ ]:
df1 = pd.DataFrame({
    'first':pd.Series([1,2],index=["A","B"]),
    'second':pd.Series([1,2],index=["A","B"])})
df2 = pd.DataFrame({
    'first':pd.Series([3,2],index=["A","C"]),
    'second':pd.Series([2,4],index=["A","C"])})

In [ ]:
display("df1", "df2" , "df1 + df2")

Pandas arithmetic functions have a `fill_value` parameter to be used in place of missing entries:

In [ ]:
df1.add(df2)

In [ ]:
df1.add(df2, fill_value=0)

In [ ]:
df1.loc['C']=0
df2.loc['B']=0
display("df1" , "df2" , "df1+df2")

### Operations Between DataFrames and Series
* Broadcasting, *similar* to NumPy
   * Limited to row-wise

In [ ]:
ints = np.random.randint(0,10,(3,4))
df = pd.DataFrame(ints, columns=['A','B','C','D'], index=['x','y','z'])
df

In [ ]:
display("df" , "df - df.loc['x']" , "df.sub(df.loc['x'], axis=1)")

In [ ]:
df

In [ ]:
# Broadcasting rules...
display("df - df['A']" , "df.sub(df['A'], axis=0)")

`pd.DataFrame(df['A'])` is a single column dataframe, but there is no broadcasting between dataframes...

In [ ]:
display("pd.DataFrame(df['A'])" , "df - pd.DataFrame(df['A'])")

## Handling Missing Data
Pandas uses *Sentinel Values* to handle not available (NA) values:
* `None`, `np.nan` and `pd.NA`
    * Integer data: `pd.NA`
    * Floating point data: `np.nan` and `pd.NA`
    * General object data: `None`, `np.nan` and `pd.NA`
* By default, in numeric data, `None` &rarr; `np.nan` (floating point) 
<br>

In [ ]:
s = pd.Series([None,2,3])
pprint('s', 's[0]', 'type(s[0])', align=True)

In [ ]:
s = pd.Series([np.nan,2,3])
pprint('s', 's[0]', 'type(s[0])', align=True)

In [ ]:
s = pd.Series([pd.NA,2,3])
pprint('s', 's[0]', 'type(s[0])', align=True)

In [ ]:
s = pd.Series([None,2,3], dtype='Int64')
pprint('s', 's[0]', 'type(s[0])', align=True)

In [ ]:
s = pd.Series([np.nan,2,3], dtype='Int64')
pprint('s', 's[0]', 'type(s[0])', align=True)

In [ ]:
pprint('pd.Series([None,"a",123])')

In [ ]:
pprint('pd.Series([np.nan,"a",123])')

In [ ]:
pprint('pd.Series([pd.NA,"a",123])')

Table of upcasting conventions in Pandas when NA values are introduced:

|Typeclass     | Conversion when storing NAs | NA sentinel value      |
|--------------|-----------------------------|------------------------|
| ``floating`` | No change                   | ``np.nan``             |
| ``object``   | No change                   | ``None`` or ``np.nan`` |
| ``integer``  | Cast to ``float64``         | ``np.nan``             |
| ``boolean``  | Cast to ``object``          | ``None`` or ``np.nan`` |

### Ignoring Null values
* Most NumPy functions are not `nan`-aware (generate `nan`)
   * `nan`-aware NumPy functions (ignore `nan`s):
      * `np.nansum`
      * `np.nanmin`
      * `np.nanmax`
   * NumPy does not handle `pd.NA` &rarr; **ERROR** 
* Pandas functions are `NaN` and `NA` aware (ignore `nan`s)

<br>

In [ ]:
s = pd.Series([1,2,None,3,4])
pprint('s.values',
       's.values.sum()', 's.values.min()', 's.values.max()',
       'np.sum(s.values)', 'np.min(s.values)', 'np.max(s.values)',
       'np.nansum(s.values)', 'np.nanmin(s.values)', 'np.nanmax(s.values)',
       's.sum()', 's.min()', 's.max()', align=True)

In [ ]:
s = pd.Series([1,2,pd.NA,3,4])
pprint('s.values',
       's.sum()', 's.min()', 's.max()', align=True)
# ERROR!
#pprint('s.values.sum()', 's.values.min()', 's.values.max()',
#       'np.sum(s.values)', 'np.min(s.values)', 'np.max(s.values)',
#       'np.nansum(s.values)', 'np.nanmin(s.values)', 'np.nanmax(s.values)')

### Detecting Null values

* `isnull` &rarr; boolean mask (`True`/`False`) where Null values
* `notnull` &rarr; opposite of `isnull`

<br>

In [ ]:
s = pd.Series([1,2,None,3,4])
pprint('s.isnull()', 's.notnull()', sep="\n\n", align=True)

In [ ]:
s[s.notnull()]

### Dropping Null values

* `dropna` &rarr; copy with the null values removed
   * Series: remove Null values
   * DataFrame: remove rows/columns containing Null values
* `inplace` parameter: bool, default False
   * If True, remove/fill in-place. Note: this will modify any other views on this object (e.g., a no-copy slice for a column in a DataFrame).

<br>

In [ ]:
s = pd.Series([1,2,None,3,4])
s.dropna() 

In [ ]:
pprint('s',end='\n\n')
s.dropna(inplace=True) 
pprint('s')

In [ ]:
a = np.arange(15,dtype='float64').reshape(3,5)
a[(0,2,2),(3,0,2)] = np.nan
df = pd.DataFrame(a)
df

In [ ]:
df.dropna() # df.dropna(axis=0)

In [ ]:
df.dropna(axis=1)

### Filling Null values

* `fillna` &rarr; copy with the null values replaced
* `inplace` parameter: bool, default False
   * If True, remove/fill in-place. Note: this will modify any other views on this object (e.g., a no-copy slice for a column in a DataFrame).
* **DEPRECATED** `method` parameter: {`bfill`, `ffill`, None}, default None
   * `ffill` &rarr; *forward fill*,  propagate the previous value forward
   * `bfill` &rarr; *back fill*,  propagate the next value backward
* use `data.ffill()` and `data.bfill()` instead
   * `axis` parameter: axis along which forward/back fill

<br>

In [ ]:
s = pd.Series([1, None, 2, None, 3], index=list('abcde'), dtype='Int64')
pprint('s', 's.fillna(0)', 's.ffill()', 's.bfill()', align=True)

In [ ]:
a = np.arange(15,dtype='float64').reshape(3,5)
a[(0,2,2),(3,0,2)] = np.nan
df = pd.DataFrame(a)

display("df" , "df.fillna(0)")

In [ ]:
df

In [ ]:
display("df.ffill()" , "df.ffill(axis=1)")

In [ ]:
df

In [ ]:
display("df.bfill()" , "df.bfill(axis=1)")

## Concatenating Datasets

In [ ]:
def make_df(col_index,row_index):
    """Quickly make a DataFrame"""
    data = {c: [str(c) + str(r) for r in row_index] for c in col_index}
    return pd.DataFrame(data, index=list(row_index), columns=list(col_index))

# example DataFrame
display("make_df('ABCDEF',range(3))" , "make_df('ABCDEF','xyz')")

### `pd.concat()`
* Similar to NumPy `concatenate`

```python
pd.concat(objs, axis=0, join='outer', ignore_index=False, keys=None,
          levels=None, names=None, verify_integrity=False,
          sort=False, copy=True)
```

In [ ]:
s1 = pd.Series(['A', 'B', 'C'], index=[1, 2, 3])
s2 = pd.Series(['D', 'E', 'F'], index=[4, 5, 6])
pd.concat([s1, s2])

In [ ]:
s1 = pd.Series(['A', 'B', 'C'], index=[1, 2, 3])
s2 = pd.Series(['D', 'E', 'F'], index=[1, 2, 3])
pd.concat([s1, s2], axis=1)

In [ ]:
df1 = make_df('AB', 'uv')
df2 = make_df('AB', 'xy')
display('df1', 'df2', 'pd.concat([df1, df2])')

In [ ]:
df1 = make_df('AB', 'xy')
df2 = make_df('CD', 'xy')
display('df1', 'df2', 'pd.concat([df1, df2], axis=1)')

* **Index preservation**: `pd.concat` *preserves* the Index (result could have duplicate indices)

In [ ]:
df1 = make_df('AB', 'xy')
df2 = make_df('AB', 'xy')
df_axis0 = pd.concat([df1, df2])
df_axis1 = pd.concat([df1, df2], axis=1)
display('df1', 'df2' , 'df_axis0' , 'df_axis1')

In [ ]:
pd.concat([df1, df2]).loc['x']

In [ ]:
pd.concat([df1, df2], axis=1)['A']

* `verify_integrity=True` prevents index duplication (raises `ValueError`)

In [ ]:
df1 = make_df('AB', 'xy')
df2 = make_df('AB', 'xy')
try:
    pd.concat([df1, df2], verify_integrity=True)
except ValueError as e:
    print("axis=0 ValueError:", e)
try:
    pd.concat([df1, df2], axis=1, verify_integrity=True)
except ValueError as e:
    print("axis=1 ValueError:", e)
display('df1', 'df2')

* `ignore_index=True` ignores overlapped indices (replaced by `RangeIndex`)

In [ ]:
df1 = make_df('AB', 'xy')
df2 = make_df('AB', 'xy')
df_axis0 = pd.concat([df1, df2], ignore_index=True)
df_axis1 = pd.concat([df1, df2], axis=1, ignore_index=True)
pprint('df_axis0.index', 'df_axis0.columns',
       'df_axis1.index', 'df_axis1.columns', align=True)
display('df1', 'df2' , 'df_axis0' , 'df_axis1')

* `join="outer"` (default): could create Null data
* `join="inner`: avoids Null data
   * `axis=0` &rarr; removes columns
   * `axis=1` &rarr; removes rows

In [ ]:
df1 = make_df('ABC', [1,2])
df2 = make_df('BCD', [3,4])
display('df1', 'df2')

In [ ]:
df_outer = pd.concat([df1, df2])
df_inner = pd.concat([df1, df2], join='inner')
display('df_outer', 'df_inner')

In [ ]:
df1 = make_df('AB', [1,2,3])
df2 = make_df('CD', [2,3,4])
display('df1', 'df2')

In [ ]:
df_outer = pd.concat([df1, df2], axis=1)
df_inner = pd.concat([df1, df2], axis=1, join='inner')
display('df_outer', 'df_inner')

## Merging Datasets

### `pd.merge()`

* Merges two Series/DataFrame
* *Common* columns are used as merging keys

```python
pd.merge(left, right, how='inner', on=None, left_on=None, right_on=None,
         left_index=False, right_index=False, sort=False,
         suffixes=('_x', '_y'), copy=None, indicator=False, validate=None)
```

In [ ]:
df1 = pd.DataFrame({'employee': ['Bob', 'Jake', 'Lisa'],
                    'group': ['Accounting', 'Engineering','Engineering']})
df2 = pd.DataFrame({'employee': ['Lisa', 'Bob', 'Jake'],
                    'hire_date': [2004, 2008, 2012]})
display('df1', 'df2')

In [ ]:
pd.merge(df1, df2)

In [ ]:
df1 = pd.merge(df1, df2)
df2 = pd.DataFrame({'group': ['Accounting', 'Engineering', 'HR'],
                    'supervisor': ['Carly', 'Guido', 'Steve']})
display('df1', 'df2')

In [ ]:
pd.merge(df1, df2)

In [ ]:
df2 = pd.DataFrame({'group': ['Accounting', 'Accounting', 'Engineering', 'Engineering'],
                    'skills': ['math', 'spreadsheets', 'software', 'math']})
display('df1', 'df2')

In [ ]:
pd.merge(df1, df2)

* `on=column_name` &rarr; use column as key
* `left_on=name1`, `right_on=name2` &rarr; use columns as key

In [ ]:
df2 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'], 'salary': [70000, 80000, 120000, 90000]})
display('df1', 'df2')

In [ ]:
pd.merge(df1, df2, left_on="employee", right_on="name")

Using `left_on=name1`, `right_on=name2` implies a redundant column

In [ ]:
pd.merge(df1, df2, left_on="employee", right_on="name")

In [ ]:
pd.merge(df1, df2, left_on="employee", right_on="name").drop('name', axis=1)

* `left_index=True`, `right_index=True` &rarr; use indices as key

In [ ]:
df2 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa'],
                    'salary': [70000, 80000, 120000]})
df2.set_index('name', inplace=True)
display('df1', 'df2')

In [ ]:
pd.merge(df1, df2, left_on='employee', right_index=True)

* `df1.join(df2)` &rarr; index based merging
   * `pd.merge(df1, df2, left_index=True, right_index=True)`

In [ ]:
df1.set_index('employee', inplace=True)
display('df1', 'df2')

In [ ]:
df1.join(df2)

* `how='inner'` (default) &rarr; intersection

In [ ]:
df1 = pd.DataFrame({'name': ['Peter', 'Paul', 'Mary', 'Adam'],
                    'food': ['fish', 'beans', 'bread' , 'fish']})
df2 = pd.DataFrame({'name': ['Mary', 'Joseph', 'Peter' , 'Alice'],
                    'drink': ['wine', 'beer' , 'water' , 'beer']})
display('df1', 'df2', 'pd.merge(df1, df2)')

* `how='outer'` &rarr; union (Null values)

In [ ]:
display('df1', 'df2', 'pd.merge(df1, df2, how="outer")')

* `how='left'` &rarr; keep left key (Null values)

In [ ]:
display('df1', 'df2', 'pd.merge(df1, df2, how="left")')

* `how='right'` &rarr; keep left key (Null values)

In [ ]:
display('df1', 'df2', 'pd.merge(df1, df2, how="right")')

## Aggregation and Grouping

### Planets Data

Information on exoplanets (planets that orbit a star other than the Sun) that astronomers have discovered.

In [ ]:
import seaborn as sns
planets = sns.load_dataset('planets')
planets.shape

In [ ]:
planets.head(10)  #planets[:10]

`7	Radial Velocity	1	798.500	NaN	21.41	1996` &rarr; **NaN values!!!**

In [ ]:
planets.isnull().values.sum()

In [ ]:
print(planets.shape)
planets.dropna(inplace=True)
print(planets.shape)

### Aggregation functions

Some built-in Pandas aggregations:

| Aggregation              | Description                     |
|--------------------------|---------------------------------|
| ``count()``              | Total number of items           |
| ``first()``, ``last()``  | First and last item             |
| ``mean()``, ``median()`` | Mean and median                 |
| ``min()``, ``max()``     | Minimum and maximum             |
| ``std()``, ``var()``     | Standard deviation and variance |
| ``mad()``                | Mean absolute deviation         |
| ``sum()`` , ``prod()``   | Sum and product of all items    |

In [ ]:
planets = sns.load_dataset('planets')
df = planets.dropna().drop(['method','number'], axis=1)
print(df.shape)
pprint('df.min()', 'df.max()', 'df.mean()', 'df.std()', align=True)

In [ ]:
df = planets.drop(['method','number'], axis=1)
print(df.shape)
pprint('df.min()', 'df.max()', 'df.mean()', 'df.std()')

* `dataframe.describe(percentiles=None, include=None, exclude=None)`: Generate descriptive statistics

In [ ]:
df.describe()

In [ ]:
planets.describe()

### The GroupBy Object
* `df.groupby()` &rarr; a *collection** of DataFrames
   * Kind of a special view of a DataFrame
* Can be indexed just like a DataFrame &rarr; a GroupBy object
* Provides easy operations of type *split-apply-combine*

<center><img src="img/split-apply-combine.png" alt="split-apply-combine" style="width: 80%;"/></center>

In [ ]:
planets.groupby('method')

In [ ]:
planets.groupby('method')['orbital_period']

In [ ]:
planets.groupby('method')['orbital_period'].median()

Methods not explicitly implemented by the GroupBy object are passed through and called on the groups.

In [ ]:
planets['orbital_period'].describe()

In [ ]:
planets.groupby('method')['orbital_period'].describe()

GroupBy object is iterable
* A sequence of `(groupby_value , Series/DataFrame)` pairs

In [ ]:
for method,df in planets.groupby('method'):
    print(f'{method:30s} {df.shape=}')

### Aggregate, filter, transform, apply
GroupBy objects have `aggregate()`, `filter()`, `transform()`, and `apply()` methods that efficiently implement a variety of useful operations before combining the grouped data.


In [ ]:
planets.groupby('method')[['orbital_period','distance']].aggregate(['count','min','max'])

## Working with Time Series

* Pandas is compatible with dates, times, and time-indexed data
* Time-based Index can be used for time series
* Standard + date-only special indexing

### Python dates and times: `datetime` and `dateutil`

In [ ]:
from datetime import datetime
date = datetime(year=2025, month=4, day=14)
date

<br>

`dateutil` module can parse dates from a variety of string formats:

In [ ]:
from dateutil import parser
date = parser.parse("14th of April, 2025")
date

### NumPy's `datetime64`

In [ ]:
import numpy as np
date = np.array('2025-04-14', dtype=np.datetime64)
pprint('date.size', 'date.shape', 'date', align=True)

<br>

Vectorized operations can be applied to dates:

In [ ]:
date + np.arange(20)

### Dates and times in Pandas

In [ ]:
date = pd.to_datetime("4th of July, 2015")
date

<br>

Vectorized operations can be applied to dates:

In [ ]:
pd.to_timedelta(np.arange(3), 'D')

In [ ]:
date + pd.to_timedelta(np.arange(20), 'D')

### Dates and times in Pandas
`DatetimeIndex` provides a time index

In [ ]:
index = pd.DatetimeIndex(['2014-07-04', '2014-08-04',
                          '2015-07-04', '2015-08-04'])
data = pd.Series([0, 1, 2, 3], index=index)
data

Standard indexing patterns:

In [ ]:
pprint("data['2014-08-04']", "data[:'2015-07-04']", "data[['2014-07-04','2015-07-04']]", align=True, sep="\n\n")

Date-only indexing patterns:

In [ ]:
pprint("data['2014']", "data[:'2015-07']", align=True, sep="\n\n")


### Time Series Data Structures
* *time stamps*: `Timestamp` (np.datetime64) &rarr; `DatetimeIndex`
* *time periods*: `Period` (np.datetime64) &rarr; `PeriodIndex`
* *time deltas or durations*: `Timedelta` (np.timedelta64) &rarr; `TimedeltaIndex`

<br>

* `pd.to_datetime()` (parser) &rarr; `Timestamp` or `DatetimeIndex`

In [ ]:
dates = pd.to_datetime([datetime(2015, 7, 3), '4th of July, 2015',
                       '2015-Jul-6', '07-07-2015', '20150708'])
pprint('dates','dates[0]', align=True)

* `DatetimeIndex.to_period()` &rarr; `PeriodIndex`

In [ ]:
dates.to_period('D')

In [ ]:
days = dates.to_period('D')
months = dates.to_period('M')
years = dates.to_period('Y')
pprint('dates[:2]', 'days[:2]', 'months[:2]', 'years[:2]', align=True)

* `Timestamp - Timestamp` &rarr; `Timedelta`:

In [ ]:
dates[1]-dates[0]

* `DatetimeIndex - Timestamp` &rarr; `TimedeltaIndex `:

In [ ]:
dates-dates[0]

* `pd.date_range()` &rarr; `DatetimeIndex`
* `pd.period_range()` &rarr; `PeriodIndex`
* `pd.timedelta_range()` &rarr; `TimedeltaIndex`

In [ ]:
pd.date_range('2015-07-03', '2015-07-10')

In [ ]:
pd.date_range('2015-07-03', periods=8, freq='h')

In [ ]:
pd.period_range('2015-07', periods=8, freq='M')

In [ ]:
pd.timedelta_range(0, periods=10, freq='h')

### Resampling, Shifting, and Windowing

In [ ]:
#!pip install pandas-datareader
from pandas_datareader import data
#goog = data.DataReader('GOOG', start=2020, end='2024',data_source='stooq')
goog = data.DataReader('GOOG', start=2020, end='2024', data_source='stooq').sort_index()
goog.head()

Some visual configurations...

In [ ]:
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn
seaborn.set()
mpl.rcParams['figure.figsize'] = (5.33,4)
mpl.rcParams['axes.labelsize'] = 10  # Example: 14 points
mpl.rcParams['xtick.labelsize'] = 8  # Example: 12 points for x-axis ticks
mpl.rcParams['ytick.labelsize'] = 8  # Example: 12 points for y-axis ticks

In [ ]:
goog = goog['Close']

In [ ]:
goog.plot(linewidth=0.5);

* `resample(freq)` &rarr; grouped object ready for a aggregation/reduction operation
* `resample(freq).aggregate_func()` &rarr; new time series with aggregated values
* `asfreq(freq)` &rarr; new time series with selected/periodic values
   * $\approx$ `resample(freq).last()`

In [ ]:
# 'BYE': Business Year End (last business day of year)
goog.plot(alpha=0.5, style='-', linewidth=0.5)
goog.resample('BYE').mean().plot(style=':') 
goog.asfreq('BYE').plot(style='--');
plt.legend(['original', 'resample', 'asfreq'], loc='upper left');

Resampling (`resample+aggregation` or `asfreq`) can generate Null values
* `method="ffill"|"bfill"` &rarr; how values are imputed

In [ ]:
d = goog.head()
pprint("d" , "d.asfreq('D')", "d.asfreq('D',method='ffill')", align=True)

In [ ]:
fig, ax = plt.subplots(2, sharex=True)
data = goog.head()

data.asfreq('D').plot(ax=ax[0], marker='o')

data.asfreq('D', method='bfill').plot(ax=ax[1], style='-o')
data.asfreq('D', method='ffill').plot(ax=ax[1], style='--o')
ax[1].legend(["back-fill", "forward-fill"]);

* `shift()` &rarr; shift the data
   *  `shift(int)` &rarr; shift the data `int` positions
   *  `shift(int, freq)` &rarr; shift the data `int` number of `freq`

In [ ]:
pprint("goog.head()", "goog.shift(3).head()", "goog.shift(3,'D').head()", align=True)

Compute differences over time (frequencies):

In [ ]:
pprint("goog.head()", "goog.shift(-365,'D').head()", align=True)

In [ ]:
data = goog.asfreq('D', method='ffill')
ROI = 100 * (data.shift(-365,'D') / data - 1)
ROI.plot()
plt.ylabel('% Return on Investment');

* `rolling()` &rarr; rolling window
   *  aggregation/reduction functions can be applied to a rolling window

In [ ]:
r365 = goog.rolling(365).mean()
r200 = goog.rolling(200).mean()
data = pd.DataFrame({'GOOG': goog, '1year mean': r365, '200d mean': r200})
ax = data.plot(style=['-', '--', ':'])
ax.lines[0].set_alpha(0.3)


`rolling(365)` is not rolling 1 year...

In [ ]:
r365 = goog.asfreq('D', method='ffill').rolling(365).mean()
r200 = goog.asfreq('D', method='ffill').rolling(200).mean()
data = pd.DataFrame({'GOOG': goog, '1year mean': r365, '200d mean': r200})
ax = data.plot(style=['-', '--', ':'])
ax.lines[0].set_alpha(0.3)